In [1]:
import pandas as pd
pd.options.display.max_columns = None  # Show all columns in the output

In [ ]:
stops = pd.read_excel(r'C:\Users\ali.etezady\OneDrive - Resource Systems Group, Inc\SANDAG\SANDAG Tour Survey_FINAL.xlsx', 
                     sheet_name='Trips')

persons = pd.read_excel(r'C:\Users\ali.etezady\OneDrive - Resource Systems Group, Inc\SANDAG\SANDAG Tour Survey_FINAL.xlsx', 
                     sheet_name='Person')

dict = pd.read_excel(r'C:\Users\ali.etezady\OneDrive - Resource Systems Group, Inc\SANDAG\SANDAG Tour Survey_FINAL.xlsx', 
                     sheet_name='Data Dictionary')

### functions

In [68]:
def create_trips_from_stops(data):
    """
    Convert stops data to trips data.
    Group by 'token' (trip chain identifier) and sort by 'trip_number' (stop sequence).
    Each trip connects two consecutive stops in the same trip chain.
    """

    # data['trip_number'] = data.groupby('token').cumcount() + 1
    trips_list = []
    
    # Group by token (person_id) and process each trip chain
    for token, trip_chain in data.groupby('token'):
        # Sort stops by trip_number to ensure correct order
        trip_chain = trip_chain.sort_values('trip_number').reset_index(drop=True)
        
        # Skip if trip chain has less than 2 stops (can't form a trip)
        if len(trip_chain) < 2:
            continue
            
        # Create trips from consecutive stops
        for i in range(len(trip_chain) - 1):
            origin_stop = trip_chain.iloc[i]
            destination_stop = trip_chain.iloc[i + 1]
            
            # Create trip record
            trip = {
                'PER_ID':       token,  # Person ID (using token as person identifier)
                'HH_ID':        'HH_' + token,      # Household ID (dummy value)
                'DAY_ID':       1,                   # Day ID (dummy value)
                'PER_NUM':      1,                   # Person number (dummy value)
                'DAYNUM':       1,                   # Day number (dummy value)
                'trip_id':      destination_stop['id'],  # Trip ID (renamed from original 'id' column)
                'trip_number':  i + 1,  # Trip number in the chain (1, 2, 3, ...)
                'orig_purpose': origin_stop['purpose_at_stop'],
                'dest_purpose': destination_stop['purpose_at_stop'],
                'travel_mode1': destination_stop['travel_mode1'],  # Primary mode used to reach destination
                'travel_mode2': destination_stop['travel_mode2'],
                'travel_mode3': destination_stop['travel_mode3'],
                'travel_mode4': destination_stop['travel_mode4'],
                'travel_mode5': destination_stop['travel_mode5'],
                'accessmode':   destination_stop['accessmode'],
                'egressmode':   destination_stop['egressmode'],
                'depart_time':  origin_stop['depart_time'],
                'arrive_time':  destination_stop['arrive_time'],
                'o_lon':        origin_stop['stop_longitude'],
                'o_lat':        origin_stop['stop_latitude'],
                'd_lon':        destination_stop['stop_longitude'],
                'd_lat':        destination_stop['stop_latitude'],
                'day_id':       1
  
            }
            
            trips_list.append(trip)
    
    # Convert to DataFrame
    trips_df = pd.DataFrame(trips_list)
    
    return trips_df


def get_representative_mode(row):
    """
    Determine the representative mode based on hierarchy.
    Returns the mode with highest priority from the combination.
    """
    modes_in_trip = []
    for i in range(1, 6):
        mode = row[f'travel_mode{i}']
        if pd.notna(mode):
            modes_in_trip.append(mode)
    
    if not modes_in_trip:
        return 'No Mode'
    
    # Find mode with highest priority
    max_priority = 0
    representative_mode = modes_in_trip[0]
    
    for mode in modes_in_trip:
        priority = MODE_HIERARCHY.get(mode, 0)
        if priority > max_priority:
            max_priority = priority
            representative_mode = mode
    
    return representative_mode

### create trips

In [14]:
trips_final = create_trips_from_stops(stops)

In [32]:
# Display basic information
print(f"Original stops data: {len(stops)} records")
print(f"Created trips data: {len(trips_final)} trips")
print(f"Number of unique trip IDs: {trips_final['trip_id'].nunique()}")
print('--------------------------------------------')
print(f"Original number of unique people: {persons['token'].nunique()}")
print(f"Number of unique people in the processed trips: {trips_final['PER_ID'].nunique()}")
missing = persons[~persons['token'].isin(trips_final['PER_ID'])].token.to_list()
stops[stops['token'].isin(missing)].did_travel.value_counts()
print(f'Appears {len(missing)} people did not travel')


Original stops data: 3328 records
Created trips data: 2302 trips
Number of unique trip IDs: 2302
--------------------------------------------
Original number of unique people: 1026
Number of unique people in the processed trips: 920
Appears 106 people did not travel


### check for modes

In [33]:
# let's examine what modes are available
print("=== AVAILABLE TRAVEL MODES ===")
for i in range(1, 6):
    unique_modes = trips_final[f'travel_mode{i}'].dropna().unique()
    print(f"travel_mode{i}: {sorted(unique_modes)}")

print("\n=== MODE COMBINATIONS ANALYSIS ===")

# Create a function to combine all travel modes into a single string for analysis
def get_mode_combination(row):
    modes = []
    for i in range(1, 6):
        mode = row[f'travel_mode{i}']
        if pd.notna(mode):
            modes.append(mode)
    return ' + '.join(modes) if modes else 'No Mode'

# Add mode combination column for analysis
trips_final['mode_combination'] = trips_final.apply(get_mode_combination, axis=1)

# Count frequency of each combination
mode_combinations = trips_final['mode_combination'].value_counts()
print(f"Total unique combinations: {len(mode_combinations)}")
print("\nTop 20 most frequent combinations:")
print(mode_combinations.head(20))

print("\n=== DETAILED COMBINATION BREAKDOWN ===")
print("Showing combinations with their component modes:")
for combo, count in mode_combinations.head(15).items():
    print(f"{combo}: {count} trips")

=== AVAILABLE TRAVEL MODES ===
travel_mode1: ['SQ001', 'SQ002', 'SQ003', 'SQ004', 'SQ005', 'SQ006', 'SQ007', 'SQ008', 'SQ009', 'SQ010', 'SQ011', 'SQ012', 'SQ013', 'SQ014', 'SQ016', 'SQ018']
travel_mode2: ['SQ001', 'SQ003', 'SQ004', 'SQ005', 'SQ006', 'SQ007', 'SQ008', 'SQ009', 'SQ010', 'SQ011', 'SQ012', 'SQ013', 'SQ016', 'SQ018']
travel_mode3: ['SQ001', 'SQ002', 'SQ003', 'SQ004', 'SQ005', 'SQ006', 'SQ007', 'SQ008', 'SQ009', 'SQ010', 'SQ011', 'SQ012', 'SQ016', 'SQ018']
travel_mode4: ['SQ001', 'SQ007', 'SQ008', 'SQ009', 'SQ010', 'SQ011', 'SQ012', 'SQ013', 'SQ014', 'SQ015', 'SQ018']
travel_mode5: ['SQ001', 'SQ003', 'SQ007', 'SQ008', 'SQ009', 'SQ010', 'SQ011', 'SQ012', 'SQ013', 'SQ014', 'SQ016', 'SQ018']

=== MODE COMBINATIONS ANALYSIS ===
Total unique combinations: 235

Top 20 most frequent combinations:
SQ001                            664
SQ010                            206
SQ008                            134
SQ011                            113
SQ007                            103
SQ0

In [ ]:
_MODE_HIERARCHY = {
    # Motorized individual transport (highest priority - represents main trip mode)
    'Drove alone and parked': 9,
    'Drove or rode with others and parked': 9,
    
    # Public transit (high priority - primary mode for longer trips)
    'Any MTS, NCTD, Coaster rail services (Trolley, Sprinter, or Coaster)': 7,
    'Any MTS or NCTD public buses': 6,
    'Was dropped off by someone I know': 5,
    'Taxi': 5, 
    'Uber, Lyft, etc ( Private)': 5,
    'Uber, lyft, (Pol and Shared)': 5,
    'Electric vehicle shuttle': 5,
    'other shuttle': 5,
    
    # Active transport - motorized (medium priority)
    'E-Bike (Personal)': 4,
    'E-Bike (Shared)': 4,
    'E-scooter (Personal)': 3,
    'E-scooter (Shared)': 3,
    'Bike (Personal)': 3,
    
    # Active transport - non-motorized (lower priority - often access/egress)
    'Skateboard': 2,
    'Wheelchair': 2,
    'Walk': 1  # Lowest priority - often just access/egress to main mode
}

mode_dict = dict[dict.Column=='travel_mode'][['Code', 'Value']].set_index('Code')['Value'].to_dict()
MODE_HIERARCHY = {k: _MODE_HIERARCHY[v] for k, v in mode_dict.items() if v in _MODE_HIERARCHY}

In [70]:
trips_final['representative_mode'] = trips_final.apply(get_representative_mode, axis=1)

In [75]:
print("=== MODE HIERARCHY ANALYSIS ===")
print("Mode priorities (higher number = higher priority):")
for mode, priority in sorted(MODE_HIERARCHY.items(), key=lambda x: x[1], reverse=True):
    count = trips_final['representative_mode'].value_counts().get(mode, 0)
    print(f"Priority {priority}: {mode} -> {count} trips")

print(f"\n=== REPRESENTATIVE MODE SUMMARY ===")
representative_counts = trips_final['representative_mode'].value_counts()
print(f"Total trips: {len(trips_final)}")
print("\nRepresentative mode distribution:")
for mode, count in representative_counts.items():
    percentage = (count / len(trips_final)) * 100
    print(f"{mode}: {count} trips ({percentage:.1f}%)")

print(f"\n=== EXAMPLE MODE COMBINATIONS AND THEIR REPRESENTATIVES ===")
# Show examples of how combinations are resolved
example_combinations = [
    'Walk + Any MTS or NCTD public buses',
    'Walk + Any MTS or NCTD public buses + Walk', 
    'Any MTS or NCTD public buses + Walk',
    'Walk + Any MTS, NCTD, Coaster rail services (Trolley, Sprinter, or Coaster)',
    'Drove alone and parked',
    'Walk + Any MTS or NCTD public buses + Any MTS, NCTD, Coaster rail services (Trolley, Sprinter, or Coaster)'
]

for combo in example_combinations:
    if combo in trips_final['mode_combination'].values:
        sample_trip = trips_final[trips_final['mode_combination'] == combo].iloc[0]
        rep_mode = sample_trip['representative_mode']
        print(f"'{combo}' -> Representative: '{rep_mode}'")

=== MODE HIERARCHY ANALYSIS ===
Mode priorities (higher number = higher priority):
Priority 9: SQ008 -> 177 trips
Priority 9: SQ009 -> 119 trips
Priority 7: SQ011 -> 429 trips
Priority 6: SQ010 -> 621 trips
Priority 5: SQ007 -> 123 trips
Priority 5: SQ012 -> 75 trips
Priority 5: SQ013 -> 0 trips
Priority 5: SQ014 -> 7 trips
Priority 5: SQ017 -> 0 trips
Priority 5: SQ018 -> 1 trips
Priority 4: SQ005 -> 6 trips
Priority 4: SQ006 -> 7 trips
Priority 3: SQ004 -> 34 trips
Priority 3: SQ015 -> 0 trips
Priority 3: SQ016 -> 14 trips
Priority 2: SQ002 -> 9 trips
Priority 2: SQ003 -> 16 trips
Priority 1: SQ001 -> 664 trips

=== REPRESENTATIVE MODE SUMMARY ===
Total trips: 2302

Representative mode distribution:
SQ001: 664 trips (28.8%)
SQ010: 621 trips (27.0%)
SQ011: 429 trips (18.6%)
SQ008: 177 trips (7.7%)
SQ007: 123 trips (5.3%)
SQ009: 119 trips (5.2%)
SQ012: 75 trips (3.3%)
SQ004: 34 trips (1.5%)
SQ003: 16 trips (0.7%)
SQ016: 14 trips (0.6%)
SQ002: 9 trips (0.4%)
SQ006: 7 trips (0.3%)
SQ014:

In [78]:
# Apply the representative mode as the main 'mode' column
trips_final['mode'] = trips_final['representative_mode']

print("=== FINAL SUMMARY: MODE ASSIGNMENT LOGIC ===")
print("""
The representative mode selection follows this hierarchy (highest to lowest priority):

1. PRIVATE VEHICLES (Priority 9-8):
   - Personal driving (alone/with others) takes precedence as the main trip mode
   - Taxi/rideshare services also high priority as they represent the primary transport
   - Being dropped off is considered equivalent to rideshare

2. PUBLIC TRANSIT (Priority 7-6):
   - Rail services (Trolley, Sprinter, Coaster) take precedence over buses
   - Transit modes take precedence over active transport when combined

3. ACTIVE MOTORIZED (Priority 4-3):
   - E-bikes and e-scooters represent significant trip modes
   - Personal vs shared versions have same priority

4. ACTIVE NON-MOTORIZED (Priority 2-1):
   - Walking often represents access/egress, so gets lowest priority
   - When walking is the only mode, it becomes the representative mode
   
EXAMPLES:
- "Walk + Bus" → Bus (transit takes precedence over access walking)
- "Bus + Rail" → Rail (rail takes precedence over bus)
- "Drive + Walk" → Drive (driving takes precedence over parking walk)
- "Walk only" → Walk (walking as main mode when it's the only option)
""")

print("=== VALIDATION: BEFORE vs AFTER ===")
print("Sample of trips showing mode assignment:")
sample_trips = trips_final[['PER_ID', 'trip_number', 'mode_combination', 'mode']].head(10)
print(sample_trips.to_string(index=False))

print(f"\n=== FINAL MODE DISTRIBUTION ===")
final_mode_counts = trips_final['mode'].value_counts()
print(f"Total trips: {len(trips_final)}")
for mode, count in final_mode_counts.items():
    percentage = (count / len(trips_final)) * 100
    print(f"{mode}: {count} trips ({percentage:.1f}%)")

# Clean up temporary columns if desired
# trips_final = trips_final.drop(['mode_combination', 'representative_mode'], axis=1)

=== FINAL SUMMARY: MODE ASSIGNMENT LOGIC ===

The representative mode selection follows this hierarchy (highest to lowest priority):

1. PRIVATE VEHICLES (Priority 9-8):
   - Personal driving (alone/with others) takes precedence as the main trip mode
   - Taxi/rideshare services also high priority as they represent the primary transport
   - Being dropped off is considered equivalent to rideshare

2. PUBLIC TRANSIT (Priority 7-6):
   - Rail services (Trolley, Sprinter, Coaster) take precedence over buses
   - Transit modes take precedence over active transport when combined

3. ACTIVE MOTORIZED (Priority 4-3):
   - E-bikes and e-scooters represent significant trip modes
   - Personal vs shared versions have same priority

4. ACTIVE NON-MOTORIZED (Priority 2-1):
   - Walking often represents access/egress, so gets lowest priority
   - When walking is the only mode, it becomes the representative mode
   
EXAMPLES:
- "Walk + Bus" → Bus (transit takes precedence over access walking)
- "Bus

In [79]:
persons

,token,collection,num_other_people,num_children,num_students,num_vehicles,age,employment,num_jobs,job_type,commute_freq,telework_freq,student,school_type,school_attend,ethnicity,ethnicity_other,race,race_other,gender,driver,income_detailed,residence,transit_freq,transit_pass_ownership,pronto_card_details,pronto_card_details_other,transit_pass_cost,deliveries,deliveries_other,tour_phone,tour_email
0,PSRIN,Spring,2,0,2,A3,1,A6,NaN,NaN,NaN,NaN,A4,SQ005,A2,SQ001,NaN,SQ005,NaN,A2,A1,A1,A1,A5,A2,A3,NaN,A1,SQ006,NaN,6504438244,1647tom@gmail.com
1,RLXFK,Spring,4,0,0,A2,2,A1,A1,A4,A1,A9,A1,NaN,NaN,SQ001,NaN,SQ005,NaN,A1,A2,A2,A1,A1,A2,97,N/a,NaN,SQ006,NaN,6082071960,a.neuenschwander89@yahoo.com
2,NYEUH,Spring,2,0,1,A2,1,A6,NaN,NaN,NaN,NaN,A2,SQ001,A2,SQ001,NaN,SQ005,NaN,A2,A1,A7,A4,A1,A2,A1,NaN,NaN,SQ002,NaN,6198319967,aldozoc@hotmail.it
3,XQKRG,Spring,3,1,1,A3,1,A2,A1,A1,A8,A9,A1,NaN,NaN,SQ002,NaN,SQ007,NaN,A1,A2,A3,A1,A4,A2,97,None,NaN,SQ006,NaN,760-532-5176,almajvasquez@gmail.com
4,OKZQS,Spring,5,1,2,A4,1,A7,NaN,NaN,NaN,NaN,A4,SQ004,A1,SQ001,NaN,SQ001,NaN,A1,A2,A7,A1,A6,A1,A6,NaN,A1,SQ006,NaN,6193339559,amayaperry02@gmail.com
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1021,VXQBN,Fall,4,1,0,A3,2,A1,A1,A3,A4,A5,A1,NaN,NaN,SQ001,NaN,SQ005,NaN,A2,A1,A6,A1,A4,A2,A1,NaN,NaN,SQ001;SQ005,NaN,NaN,NaN
1022,OZWVR,Fall,2,1,0,A3,2,A1,A1,A3,A3,A6,A1,NaN,NaN,SQ001,NaN,SQ005,NaN,A2,A1,A9,A1,A3,A2,A1,NaN,NaN,SQ001;SQ005,NaN,NaN,NaN
1023,RJKSX,Fall,1,1,0,A1,5,A1,A1,A1,A1,A9,A1,NaN,NaN,SQ001,NaN,SQ005,NaN,A2,A1,A2,A1,A1,A2,A1,NaN,NaN,SQ001;SQ002,NaN,NaN,NaN
1024,NWQUC,Fall,7,1,0,A5,8,A4,A1,A4,A7,A3,A1,NaN,NaN,SQ001,NaN,SQ005,NaN,A2,A1,A2,A1,A7,A3,NaN,NaN,NaN,SQ001;SQ002,NaN,NaN,NaN
